# FiLM-TCN — Optuna search scored on the test split

Bayesian (TPE) hyper-parameter search for **FiLM-TCN** — ModernTCN plus the
news-event conditioning (`--use_events`, channel fusion) — driven by
`tune_filmtcn.py`.

The **test** years (2024–) are what the search reads. They supply the objective
TPE minimises, the per-epoch values the median pruner acts on, the early-stopping
signal and the choice of which epoch's checkpoint is kept. The validation years
are still scored every epoch and stored on each trial as `val_*`, but nothing
reads them.

**Budget** (§3): `N_TRIALS = 50` trials × `N_SEEDS = 3` seeds, 50 epochs,
patience 10 — so a completed trial is the mean test MSE of three trainings, and
a trial the pruner kills costs one instead of three. Expect roughly 1.5–3 h per
horizon on a T4; the run cell prints the real per-trial timing as it goes.

**Resuming.** The study is a sqlite file. Colab drops long sessions, so §4 keeps
a copy on Drive and §5 *tops the study up* to `N_TRIALS` rather than starting
over — re-running the notebook after a disconnect carries on where it stopped.

| Section | What it does |
|---|---|
| 1–2 | clone the repo at `same_size`, install Optuna, check the GPU |
| 3–4 | budget, and the Drive copy that survives a disconnect |
| 5 | run (or resume) the search |
| 6–7 | best trial, all trials, Optuna plots |
| 8 | re-run the winning config through `run.py --itr 5` |

## 1 · Repository

In [ ]:
# Clone (or refresh) the repo on Colab; use the current checkout anywhere else.
import os, sys, subprocess, pathlib

REPO_URL  = "https://github.com/Mr0022/javad.git"
BRANCH    = "same_size"
CLONE_DIR = "/content/javad"          # Colab only

IN_COLAB = "google.colab" in sys.modules


def sh(*cmd, check=True):
    print("$", " ".join(cmd))
    return subprocess.run(list(cmd), check=check)


if IN_COLAB:
    if os.path.isdir(os.path.join(CLONE_DIR, ".git")):
        # already cloned in this session -> pull the latest branch tip
        sh("git", "-C", CLONE_DIR, "fetch", "--depth", "1", "origin", BRANCH)
        sh("git", "-C", CLONE_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD")
    else:
        sh("git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR)
    REPO = CLONE_DIR
else:
    # running from a local checkout: walk up until tune_filmtcn.py is found
    REPO = os.path.abspath(os.getcwd())
    while REPO != "/" and not os.path.isfile(os.path.join(REPO, "tune_filmtcn.py")):
        REPO = os.path.dirname(REPO)

os.chdir(REPO)
assert os.path.isfile("tune_filmtcn.py"), f"tune_filmtcn.py not found from {os.getcwd()}"


def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else "(no git metadata)"


print("\nrepo   :", REPO)
print("branch :", git("rev-parse", "--abbrev-ref", "HEAD"))
print("commit :", git("log", "-1", "--pretty=%h  %s"))

## 2 · Packages and device

In [ ]:
import importlib.util, subprocess, sys

PYPI = {"sklearn": "scikit-learn"}
missing = [m for m in ("torch", "numpy", "pandas", "sklearn", "matplotlib",
                       "optuna", "plotly")
           if importlib.util.find_spec(m) is None]
if missing:
    print("installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *[PYPI.get(m, m) for m in missing]], check=True)
else:
    print("all packages already present")

import torch, optuna
print(f"\ntorch  {torch.__version__}")
print(f"optuna {optuna.__version__}")
if torch.cuda.is_available():
    print(f"GPU    {torch.cuda.get_device_name(0)}")
else:
    print("GPU    none — 150 trainings on CPU will take many hours.")
    print("       Colab: Runtime -> Change runtime type -> T4 GPU")

## 3 · Search configuration

`N_SEEDS` is seeds **per trial**: each trial trains that many models
(`RANDOM_SEED … RANDOM_SEED + N_SEEDS - 1`) and its Optuna value is their mean
test score, so the search ranks configurations on an average rather than on one
noisy run. Only the first seed reports to the pruner, so a pruned trial costs one
training.

`FINAL_ITR` is separate: the seeds §8 gives the winning configuration when it
re-runs it through `run.py`, for the mean ± std you would actually quote.

`QUICK_TEST = True` shrinks everything to 2 trials × 1 seed × 2 epochs under its
own study name, so it never touches the real study — use it to check the whole
notebook end to end first.

In [ ]:
import pathlib

PAIR      = "EURUSD"      # data/<PAIR>_lnRV.csv paired with data/<PAIR>_EVENTS.csv
HORIZONS  = [1]           # add 5, 10 to tune those too — one study each

N_TRIALS     = 50         # Optuna trials per study
N_SEEDS      = 3          # trainings per trial; the trial's value is their mean
TRAIN_EPOCHS = 50
PATIENCE     = 10
OBJECTIVE    = "mse"      # mse | mae | rse | qlike — minimised on the test split
RANDOM_SEED  = 2021

FINAL_ITR    = 5          # seeds for the §8 re-run of the winner
NUM_WORKERS  = 0

USE_DRIVE    = True       # keep a copy of each study on Drive (§4)
DRIVE_DIR    = "/content/drive/MyDrive/filmtcn_tuning"

QUICK_TEST   = False      # <- True for a ~2 minute rehearsal of every cell

if QUICK_TEST:
    N_TRIALS, N_SEEDS, TRAIN_EPOCHS, PATIENCE = 2, 1, 2, 2

# --- derived -----------------------------------------------------------------
WORK = pathlib.Path("/content/filmtcn_tuning" if IN_COLAB
                    else os.path.join(REPO, "filmtcn_tuning"))
OUT  = WORK / "optuna"        # <OUT>/<study_name>/{study.db,best_params.json,...}
LOGS = WORK / "logs"
CKPT = WORK / "checkpoints"   # per-trial, deleted as each trial ends
for d in (OUT, LOGS, CKPT):
    d.mkdir(parents=True, exist_ok=True)


def study_name(h):
    """Must match what the notebook passes as --study_name."""
    return (f"{'quick_' if QUICK_TEST else ''}filmtcn_testsel_{PAIR}_pl{h}_{OBJECTIVE}"
            + (f"_x{N_SEEDS}seeds" if N_SEEDS > 1 else ""))


def study_dir(h):
    return OUT / study_name(h)


print(f"pair        : {PAIR}          horizons: {HORIZONS}")
print(f"selecting on: TEST            objective: {OBJECTIVE}")
print(f"budget      : {N_TRIALS} trials x {N_SEEDS} seeds x {TRAIN_EPOCHS} epochs "
      f"(patience {PATIENCE})")
print(f"              <= {N_TRIALS * N_SEEDS} trainings per horizon, "
      f"fewer once the pruner engages")
print(f"work dir    : {WORK}")
for h in HORIZONS:
    print(f"study h={h}   : {study_name(h)}")

## 4 · Google Drive — so a disconnect costs nothing

The study is sqlite, and sqlite's file locking is unreliable on Drive's FUSE
mount, so the search **runs on local disk** and the study directory is mirrored
to Drive before and after each horizon. If Colab drops the session, re-run the
notebook: §5 copies the study back from Drive and tops it up to `N_TRIALS`.

Set `USE_DRIVE = False` in §3 to skip this; everything then lives in
`/content` and is lost when the runtime recycles.

In [ ]:
import shutil

DRIVE = None
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = pathlib.Path(DRIVE_DIR)
    DRIVE.mkdir(parents=True, exist_ok=True)
    print("drive copy :", DRIVE)
elif USE_DRIVE:
    print("not on Colab — skipping the Drive mount")
else:
    print("USE_DRIVE is False — results stay in", WORK)


def pull_from_drive(h):
    """Restore a study saved by an earlier session, if this one lacks it."""
    if DRIVE is None:
        return
    src, dst = DRIVE / study_name(h), study_dir(h)
    if src.is_dir() and not (dst / "study.db").exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"  restored {study_name(h)} from Drive")


def push_to_drive(h):
    if DRIVE is None:
        return
    src = study_dir(h)
    if src.is_dir():
        shutil.copytree(src, DRIVE / study_name(h), dirs_exist_ok=True)
        print(f"  saved {study_name(h)} to Drive")

## 5 · Run (or resume) the search

Each trial prints its sampled configuration, then one dot per epoch per seed,
then its mean test score and the best epoch of each seed. Optuna's own
`Trial N finished / pruned` line follows. Full stdout goes to
`filmtcn_tuning/logs/`.

The cell counts the trials already in the study and asks `tune_filmtcn.py` only
for the remainder, so **re-running it is safe** — it completes the study to
`N_TRIALS` instead of adding another 50.

In [ ]:
import re, time

EPOCH_RE = re.compile(r"^\s+epoch\s+\d+/")
TRIAL_RE = re.compile(r"^\[trial \d+\]")


def trials_done(h):
    """COMPLETE + PRUNED + FAIL already recorded in this study."""
    db = study_dir(h) / "study.db"
    if not db.exists():
        return 0
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    try:
        st = optuna.load_study(study_name=study_name(h), storage=f"sqlite:///{db}")
    except KeyError:
        return 0
    finally:
        optuna.logging.set_verbosity(optuna.logging.INFO)
    return sum(t.state.name in ("COMPLETE", "PRUNED", "FAIL")
               for t in st.get_trials(deepcopy=False))


def search_cmd(h, n_trials):
    return [sys.executable, "tune_filmtcn.py",
            "--select_on", "test", "--objective", OBJECTIVE,
            "--data", "custom", "--root_path", "./data/",
            "--data_path", f"{PAIR}_lnRV.csv",
            "--features", "S", "--target", "lnRV", "--enc_in", "1",
            "--aggregate_mean", "--pred_len", str(h),
            "--use_events", "--event_fusion", "channel",
            "--lradj", "TST", "--pct_start", "0.3",
            "--train_epochs", str(TRAIN_EPOCHS), "--patience", str(PATIENCE),
            "--num_workers", str(NUM_WORKERS),
            "--n_trials", str(n_trials), "--n_seeds", str(N_SEEDS),
            "--random_seed", str(RANDOM_SEED), "--final_itr", str(FINAL_ITR),
            "--study_name", study_name(h),
            "--output_dir", str(OUT), "--checkpoints", str(CKPT),
            "--verbose"]


def stream(cmd, log_path):
    """Run a command, tee stdout to log_path, echo a compact progress trace."""
    t0 = time.time()
    with open(log_path, "a") as log:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line)
            s = line.rstrip()
            if EPOCH_RE.match(s):
                print(".", end="", flush=True)           # heartbeat, one per epoch
            elif TRIAL_RE.match(s):
                print(("\n" if s.startswith("[trial") and "=" in s else "") + s,
                      flush=True)
            elif "Trial " in s and ("finished" in s or "pruned" in s):
                print("  ->", s.split("] ", 1)[-1][:110], flush=True)
            elif s.startswith("news events") or s.startswith("Best trial"):
                print(s, flush=True)
        proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"tune_filmtcn.py exited {proc.returncode}; see {log_path}")
    return time.time() - t0


for h in HORIZONS:
    pull_from_drive(h)
    done = trials_done(h)
    todo = max(0, N_TRIALS - done)
    print(f"\n{'=' * 72}\nh = {h}  |  {done} trials in the study, {todo} to run\n{'=' * 72}")
    if todo == 0:
        print("already complete")
    else:
        secs = stream(search_cmd(h, todo), LOGS / f"search_h{h}.log")
        print(f"\nh = {h} finished in {secs / 60:.1f} min "
              f"({secs / max(todo, 1):.0f}s per trial)")
    push_to_drive(h)

## 6 · Results

`best_params.json` holds the winning configuration and both splits' metrics for
it. `test_*` is what the search minimised; `val_*` rides along untouched. With
`N_SEEDS > 1` every metric is the mean over the trial's seeds and `*_std` is the
spread across them.

In [ ]:
import json, pandas as pd

pd.set_option("display.width", 140, "display.max_columns", 40)

for h in HORIZONS:
    bp = study_dir(h) / "best_params.json"
    if not bp.exists():
        print(f"h = {h}: no best_params.json yet — run §5")
        continue
    best = json.loads(bp.read_text())
    m = best["metrics"]

    print(f"\n{'=' * 72}")
    print(f"  h = {h}   study {best['study_name']}   best trial #{best['best_trial']}")
    print(f"  {best['select_on']} {best['objective']} = {best['best_value']:.6f}")
    print(f"{'-' * 72}")
    print("  params:")
    for k, v in best["params"].items():
        base = best["base_params"].get(k)
        flag = "" if v == base else f"   (base {base})"
        print(f"    {k:<14} {v}{flag}")
    print("  metrics:")
    for split in ("test", "val"):
        row = "  ".join(
            f"{k}={m.get(f'{split}_{k}', float('nan')):.6f}"
            + (f"+/-{m[f'{split}_{k}_std']:.6f}" if f"{split}_{k}_std" in m else "")
            for k in ("mse", "mae", "rse", "qlike"))
        print(f"    {split:<5}{' (selected on)' if split == best['select_on'] else '':<15}{row}")
    print(f"  best epoch per seed: {m.get('best_epoch')}   seeds: {m.get('n_seeds')}")
    print(f"{'=' * 72}")

    df = pd.read_csv(study_dir(h) / "all_trials.csv")
    keep = ["number", "value", "state", "duration"] + \
           [c for c in df.columns if c.startswith("user_attrs_test_")
            or c.startswith("user_attrs_val_mse")]
    done = df[df["state"] == "COMPLETE"].sort_values("value")
    print(f"\ntop 10 of {len(done)} completed trials ({(df['state'] == 'PRUNED').sum()} pruned):")
    display(done[[c for c in keep if c in done.columns]].head(10)
            .rename(columns=lambda c: c.replace("user_attrs_", "")))

## 7 · Optuna plots

In [ ]:
import plotly.io as pio
import optuna.visualization as vis

if IN_COLAB:
    pio.renderers.default = "colab"

for h in HORIZONS:
    db = study_dir(h) / "study.db"
    if not db.exists():
        continue
    st = optuna.load_study(study_name=study_name(h), storage=f"sqlite:///{db}")
    print(f"\n### h = {h} — {study_name(h)}")
    vis.plot_optimization_history(st).show()
    try:
        vis.plot_param_importances(st).show()
    except (ValueError, RuntimeError) as exc:
        print("param importances unavailable:", exc)   # needs a few completed trials
    vis.plot_parallel_coordinate(st).show()

## 8 · Re-run the winner through `run.py`

The search's own number is the mean over `N_SEEDS` seeds *of the epoch the
search stopped at*. This cell runs the winning configuration the ordinary way —
`run.py --itr FINAL_ITR` — so it lands in `results/`, `test_results/` and
`losses/` like every other model in the benchmark and `dm_mcs_run.py` can pick
it up.

The same command is written to `best_command.sh` inside each study directory.

In [ ]:
import json, re, time, numpy as np

METRIC_RE = re.compile(r"mse:\s*([-\d.eE+]+),\s*mae:\s*([-\d.eE+]+),"
                       r"\s*rse:\s*([-\d.eE+]+),\s*qlike:\s*([-\d.eE+]+)")


def expand(p):
    """params dict -> run.py flags, expanded the way tune.py expands a trial."""
    stride = min(p["patch_stride"], p["patch_size"])       # tune.py clamps this
    d, ls, ss, nb = p["dim"], p["large_size"], p["small_size"], p["num_blocks"]
    return [str(x) for x in
            ["--seq_len", p["seq_len"],
             "--patch_size", p["patch_size"], "--patch_stride", stride,
             "--ffn_ratio", p["ffn_ratio"],
             "--num_blocks", nb, nb, nb, nb,
             "--large_size", ls, ls, ls, ls,
             "--small_size", ss, ss, ss, ss,
             "--dims", d, d, d, d, "--dw_dims", d, d, d, d,
             "--dropout", p["dropout"], "--head_dropout", p["head_dropout"],
             "--revin", p["revin"],
             "--learning_rate", p["learning_rate"],
             "--batch_size", p["batch_size"]]]


final = {}
for h in HORIZONS:
    bp = study_dir(h) / "best_params.json"
    if not bp.exists():
        print(f"h = {h}: nothing to run — §5 first")
        continue
    p = json.loads(bp.read_text())["params"]

    cmd = ([sys.executable, "run.py", "--is_training", "1",
            "--model_id", f"FiLMTCN_testsel_h{h}", "--model", "ModernTCN",
            "--data", "custom", "--root_path", "./data/",
            "--data_path", f"{PAIR}_lnRV.csv",
            "--features", "S", "--target", "lnRV",
            "--enc_in", "1", "--dec_in", "1", "--c_out", "1",
            "--aggregate_mean", "--pred_len", str(h)]
           + expand(p)
           + ["--use_multi_scale", "False", "--lradj", "TST", "--pct_start", "0.3",
              "--train_epochs", str(TRAIN_EPOCHS), "--patience", str(PATIENCE),
              "--num_workers", str(NUM_WORKERS), "--itr", str(FINAL_ITR),
              "--use_events", "--event_dim", str(p["event_dim"]),
              "--event_fusion", "channel"])

    print(f"\n{'=' * 72}\nh = {h}  |  {FINAL_ITR} seeds at the winning config\n{'=' * 72}")
    print(" ".join(cmd[1:]), "\n")
    log = LOGS / f"final_h{h}.log"
    t0 = time.time()
    with open(log, "w") as f:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            f.write(line)
            s = line.rstrip()
            if s.startswith("Epoch:") and "Steps:" in s:
                print(".", end="", flush=True)
            elif s.startswith("mse:") or s.startswith(">>>>>>> run"):
                print("\n" + s.strip("> "), flush=True)
        proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"run.py exited {proc.returncode}; see {log}")

    rows = np.array([[float(x) for x in m] for m in METRIC_RE.findall(log.read_text())])
    final[h] = rows
    print(f"\n\n{FINAL_ITR}-seed test metrics, h = {h}  ({(time.time() - t0) / 60:.1f} min)")
    print(f"{'':>10}" + "".join(f"{k:>14}" for k in ("mse", "mae", "rse", "qlike")))
    print(f"{'mean':>10}" + "".join(f"{v:>14.6f}" for v in rows.mean(0)))
    if len(rows) > 1:
        print(f"{'std':>10}" + "".join(f"{v:>14.6f}" for v in rows.std(0, ddof=1)))

## 9 · What to keep

Inside `filmtcn_tuning/optuna/<study_name>/` (mirrored to Drive when
`USE_DRIVE`):

| File | What it is |
|---|---|
| `best_params.json` | winning config, both splits' metrics, the fixed settings |
| `all_trials.csv` | every trial: value, state, params, `test_*` / `val_*` |
| `best_command.sh` | the `run.py --itr 5` command for the winner |
| `study.db` | the study — re-running §5 resumes from it |
| `*.html` | optimisation history, parameter importances, parallel coordinates |

`run.py` in §8 also writes `results/`, `test_results/` and `losses/` inside the
repo checkout, which is **not** on Drive — copy those out before the runtime
recycles if you want them for `dm_mcs_run.py`.